# Time dependent ITER simulation

In [ ]:
import Pkg; Pkg.activate("iter_test")
Pkg.add("Plots")
Pkg.develop("FUSE")
Pkg.develop("IMAS")
Pkg.develop("TurbulentTransport")
Pkg.resolve()
using Plots;
using FUSE
FUSE.ProgressMeter.ijulia_behavior(:clear);

In [ ]:
reload = false
if reload
    dd, ini, act = FUSE.load("iter_stationary");
else
    ini, act = FUSE.case_parameters(:ITER; init_from=:ods);
    dd = IMAS.dd()
    FUSE.init(dd, ini, act)
    
    ini, _ = FUSE.case_parameters(:ITER; init_from=:scalars, time_dependent=true);

    ini.time.pulse_shedule_time_basis = range(0, 300; step=1.0)
    ini.time.simulation_start = 50.0;

    rampup_ends = 10.0

    ini.rampup.side = :lfs
    ini.rampup.ends_at = rampup_ends
    ini.rampup.diverted_at = rampup_ends * 0.8

    ini.equilibrium.pressure_core = t -> ramp(t / rampup_ends) .^ 2 * 0.643e6
    ini.equilibrium.ip = t -> ramp(t / rampup_ends) * 14E6 + ramp((t - 100) / 100) * 1E6

    ini.nb_unit[1].power_launched = t -> (1 .+ ramp((t - 100) / 100)) * 16.7e6
    ini.ec_launcher[1].power_launched = t -> (1 .+ ramp((t - 100) / 100)) * 10E6
    ini.ic_antenna[1].power_launched = t -> (1 .+ ramp((t - 100) / 100)) * 12E6
    ini.lh_antenna[1].power_launched = t -> (1 .+ ramp((t - 100) / 100)) * 5E6
    ini.pellet_launcher[1].frequency = t -> (1 .+ ramp((t - 100) / 100)) * 0.01 # Hz

    # the same ip(t) can be defined with unit pulse shaping functions...
    ini.equilibrium.ip = t -> ramp(t / 10.0) * 13E6 + ramp((t - 100) / 100) * 2E6;

    # ...or by a `sequence(t, t_y_tuple_sequence)`
    ini.equilibrium.ip = t -> sequence(t, [(0.0, 0.0), (10.0, 13.0E6), (100.0, 13.0E6), (200.0, 15.0E6)]);
    
    FUSE.init(dd, ini, act; initialize_hardware=false);

    act.ActorStationaryPlasma.convergence_error = 2E-2
    act.ActorStationaryPlasma.max_iterations = 5

    act.ActorSteadyStateCurrent.current_relaxation_radius = 0.2

    act.ActorFluxMatcher.verbose = true
    act.ActorFluxMatcher.relax = 0.5

    FUSE.ActorStationaryPlasma(dd, act; verbose=false)
    
end
@checkin :stationary dd ini act;

In [ ]:
@checkout :stationary dd ini act;

# For people interested in controls, the FuseExchangeProtocol can be used run a co-simulation with a controller external to FUSE
#IMAS.fxp_connect(dd)

act.ActorDynamicPlasma.Nt = 60
act.ActorDynamicPlasma.Δt = 300.0

act.ActorDynamicPlasma.evolve_current = true
act.ActorDynamicPlasma.evolve_equilibrium = true
act.ActorDynamicPlasma.evolve_transport = true
act.ActorDynamicPlasma.evolve_hcd = true
act.ActorDynamicPlasma.evolve_pf_active = false
act.ActorDynamicPlasma.evolve_pedestal = true

act.ActorDynamicPlasma.ip_controller = true
act.ActorDynamicPlasma.time_derivatives_sources = true

actor = FUSE.ActorDynamicPlasma(dd, act; verbose=true);

@checkin :time_dep_dt dd ini act actor;

In [ ]:
Pkg.status(; mode=Pkg.PKGMODE_MANIFEST)